In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_data = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

# EDA

In [ ]:
data.shape

In [ ]:
data.head()

Example of what is not disaster tweet

In [ ]:
data[data["target"] == 0]["text"].values[10]

Example of what is disaster tweet

In [ ]:
data[data["target"] == 1]["text"].values[10]

In [ ]:
missing_table = pd.DataFrame({
    'Missing Values': data.isna().sum(),
    'Percentage (%)': (data.isnull().mean() * 100).round(2)
})

print(missing_table.sort_values(by='Missing Values', ascending=False))

In [ ]:
data.nunique()

In [ ]:
data['target'].value_counts().plot.pie(autopct='%1.1f%%', startangle=90, explode=(0,0.1), colors=['#4c72b0', '#aec7e8'])
plt.title('Target distribution')
plt.ylabel('')
plt.show()

# Model building

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import RidgeClassifier, LogisticRegression
import torch
import torch.nn as nn
import transformers
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from collections import Counter
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

In [ ]:
X = data.drop('target', axis=1)
y = data['target']

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
bert_model = AutoModel.from_pretrained("vinai/bertweet-base")
tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=False)

In [ ]:
for param in bert_model.parameters():
    param.requires_grad = True

In [ ]:
X_train_token = tokenizer(X_train["text"].tolist(), return_tensors='pt', padding=True, truncation=True, max_length=512)
X_val_token = tokenizer(X_val["text"].tolist(), return_tensors='pt', padding=True, truncation=True, max_length=512)

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        if labels is not None:
          self.labels = list(labels)
        else:
          self.labels = None

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
          item["labels"] = torch.tensor(self.labels[idx])
        return item

In [ ]:
train_dataset = TweetDataset(X_train_token, y_train)
val_dataset = TweetDataset(X_val_token, y_val)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
class Tweetmodel(nn.Module):
  def __init__(self, bert_model, hidden_dim1, hidden_dim2, output_dim):
    super(Tweetmodel, self).__init__()
    self.bert = bert_model
    embedding_dim = bert_model.config.hidden_size

    self.out = nn.Sequential(
        nn.Linear(embedding_dim, hidden_dim1),
        nn.Dropout(0.3),
        nn.ReLU(),        
        nn.Linear(hidden_dim1, hidden_dim2),
        nn.Dropout(0.2),
        nn.Linear(hidden_dim2, output_dim)
        )

  def forward(self, input_ids, attn_mask):
    bert_output = self.bert(input_ids=input_ids, attention_mask=attn_mask)
    output = bert_output.last_hidden_state[:, 0, :]
    output = self.out(output)
    return output

In [ ]:
model = Tweetmodel(bert_model, hidden_dim1=128, hidden_dim2=64, output_dim=1)

In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2, eps = 1e-7)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

counter = Counter(y)
weights = torch.tensor([counter[0] / counter[1]], dtype=torch.float).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=weights)

total_steps = len(train_loader) * 15
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

In [ ]:
for epoch in range(10):
    model.train()
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).float().to(device)

        outputs = model(input_ids=input_ids, attn_mask=attention_mask)

        loss = loss_fn(outputs, labels)
        loss.backward()

        optimizer.step()
        scheduler.step() 
        optimizer.zero_grad()
    print(f"Epoch {epoch+1} complete")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].unsqueeze(1).float().to(device)

            outputs = model(input_ids=input_ids, attn_mask=attention_mask)
            probs = torch.sigmoid(outputs)
            preds = torch.round(probs)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f"Validation Accuracy: {(correct / total) * 100:.2f}%")

# Submission

In [ ]:
test_token = tokenizer(test_data["text"].tolist(), return_tensors='pt', padding=True, truncation=True)

In [ ]:
test_dataset = TweetDataset(test_token)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
model.eval()
predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attn_mask=attention_mask)
        probs = torch.sigmoid(outputs)
        preds = torch.round(probs)
        predictions.extend(preds.squeeze().cpu().numpy().astype(int))

In [ ]:
submission = pd.DataFrame({
    'id': test_data["id"],
    'target': predictions
})

In [ ]:
submission

In [ ]:
submission.to_csv('submission.csv', index=False)